# Task 4
This serves as a template which will guide you through the implementation of this task. It is advised to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [34]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

Depending on your approach, you might need to adapt the structure of this template or parts not marked by TODOs.
It is not necessary to completely follow this template. Feel free to add more code and delete any parts that are not required.

In [35]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
BATCH_SIZE = 16
NUM_EPOCHS = 3
LR = 2e-5

train_val = pd.read_csv("train.csv")
test_val = pd.read_csv("test_no_score.csv")

# Combine title and sentence (fillna guards against the 1 null title in train.csv)
train_val["title"] = train_val["title"].fillna("").astype(str)
train_val["sentence"] = train_val["sentence"].fillna("").astype(str)
test_val["title"] = test_val["title"].fillna("").astype(str)
test_val["sentence"] = test_val["sentence"].fillna("").astype(str)

train_val["text"] = train_val["title"] + " [SEP] " + train_val["sentence"]
train_val["label"] = train_val["score"]
test_val["text"] = test_val["title"] + " [SEP] " + test_val["sentence"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Using device: mps


In [36]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels=None):
        # Pre-tokenize all texts once to avoid repeated work in __getitem__
        encoding = tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
        )
        self.input_ids = torch.tensor(encoding["input_ids"])
        self.attention_mask = torch.tensor(encoding["attention_mask"])
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return self.input_ids.shape[0]

    def __getitem__(self, index):
        sample = {
            "input_ids": self.input_ids[index],
            "attention_mask": self.attention_mask[index],
        }
        if self.labels is not None:
            sample["labels"] = self.labels[index]
        return sample

In [37]:
train_dataset = SentimentDataset(train_val["text"].tolist(), train_val["label"].tolist())
test_dataset = SentimentDataset(test_val["text"].tolist())

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True,
                          num_workers=0,
                          pin_memory=False)
test_loader = DataLoader(dataset=test_dataset,
                         batch_size=BATCH_SIZE,
                         shuffle=False,
                         num_workers=0,
                         pin_memory=False)

print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

Train batches: 782 | Test batches: 63


In [38]:
class SentimentClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        # All parameters trainable — full fine-tuning
        hidden_size = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_size, 2)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_embedding)


model = SentimentClassifier().to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable parameters: 66,364,418 / 66,364,418


In [39]:
criterion = nn.CrossEntropyLoss()

# Discriminative LR: lower for encoder, higher for classifier head
optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": LR},
    {"params": model.classifier.parameters(), "lr": LR * 10}
], weight_decay=0.01)

total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer,
                                             num_warmup_steps=warmup_steps,
                                             num_training_steps=total_steps)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1} — avg loss: {epoch_loss / len(train_loader):.4f}")

Epoch 1/3: 100%|██████████| 782/782 [06:56<00:00,  1.88it/s]


Epoch 1 — avg loss: 0.3194


Epoch 2/3: 100%|██████████| 782/782 [06:48<00:00,  1.91it/s]


Epoch 2 — avg loss: 0.1510


Epoch 3/3: 100%|██████████| 782/782 [06:47<00:00,  1.92it/s]

Epoch 3 — avg loss: 0.0710


In [40]:
model.eval()
with torch.no_grad():
    results = []
    for batch in tqdm(test_loader, desc="Inference"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(logits, dim=1).cpu().numpy()
        results.append(predictions)

    with open("result.txt", "w") as f:
        for val in np.concatenate(results):
            f.write(f"{val}\n")

print(f"Saved {len(np.concatenate(results))} predictions to result.txt")

Inference: 100%|██████████| 63/63 [00:09<00:00,  6.86it/s]

Saved 1000 predictions to result.txt
